In [2]:
import os
import json
from mlx_lm import load, generate
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# --- 1. SYSTEM PATH DEFINITIONS ---
PD_BASE = "/Volumes/Macintosh PD/Gemma_Brain"
DB_DIR = os.path.join(PD_BASE, "chroma_db")
CODE_DIR = os.path.join(PD_BASE, "code_repository")
HISTORY_FILE = os.path.join(PD_BASE, "chat_history", "session.json")

# --- 2. LIGHTWEIGHT EMBEDDING ENGINE ---
# We use a tiny CPU/M4-optimized model for embeddings to save Unified RAM
print("Initializing Memory Embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# --- 3. VECTOR DATABASE INITIALIZATION (Reading from Pendrive) ---
print("Connecting to Vector Storage on Macintosh PD...")
vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

def ingest_code_to_pendrive():
    """Reads raw code files from pendrive, chunks them, and saves to Vector DB"""
    # Specifically targeting C, C++, and Python files
    loader = DirectoryLoader(CODE_DIR, glob="**/*.*(c|cpp|py|ipynb)")
    documents = loader.load()
    
    # Split code intelligently (aware of functions/classes)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, 
        chunk_overlap=200,
        separators=["\nclass ", "\ndef ", "\nint main", "\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    
    # Persist back to Macintosh PD
    vector_db.add_documents(chunks)
    print(f"Ingested {len(chunks)} logic chunks into Macintosh PD.")

# --- 4. MLX GEN-ENGINE (Internal SSD & RAM) ---
print("Loading Gemma Model into M4 Unified Memory (8-bit)...")
# Point this to the local mlx_model folder created by the conversion step
model, tokenizer = load("mlx_model") 

def coding_assistant(query, use_rag=True):
    """The main DeepMind-style retrieval and generation loop."""
    
    context = ""
    if use_rag:
        # Search the pendrive DB for relevant code chunks
        docs = vector_db.similarity_search(query, k=3)
        context = "Reference Code Context from Macintosh PD:\n"
        for d in docs:
            context += f"--- {d.metadata.get('source', 'Unknown File')} ---\n{d.page_content}\n\n"
            
    # Construct the prompt to manage Context vs Instruction
    prompt = f"<bos><start_of_turn>user\n"
    if context:
        prompt += f"{context}\n"
    prompt += f"System Task: You are an expert C/C++/Python coding assistant.\nUser Query: {query}<end_of_turn>\n<start_of_turn>model\n"

    print("Thinking...")
    
    # MLX Generation - Fast Apple Silicon inference
    # max_tokens limits how much KV cache is used.
    response = generate(model, tokenizer, prompt=prompt, verbose=False, max_tokens=1024)
    
    # Save to history file on pendrive to maintain state without using RAM
    save_chat_history(query, response)
    
    return response

def save_chat_history(query, response):
    history =[]
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, "r") as f:
            history = json.load(f)
    history.append({"user": query, "model": response})
    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=4)

# --- EXECUTION ---
# Uncomment to ingest new code files dropped into the pendrive:
# ingest_code_to_pendrive()

# Test the Assistant
test_query = "Write a highly optimized C++ implementation of Dijkstra's algorithm using a priority queue."
output = coding_assistant(test_query, use_rag=False)
print("\n[GEMMA RESPONSE]:\n", output)

Initializing Memory Embeddings...


/var/folders/kh/qc595q4s4fxfxqgxjtcw1hkr0000gn/T/ipykernel_7597/1758916174.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6140.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to Vector Storage on Macintosh PD...


/var/folders/kh/qc595q4s4fxfxqgxjtcw1hkr0000gn/T/ipykernel_7597/1758916174.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)


Loading Gemma Model into M4 Unified Memory (8-bit)...
Thinking...

[GEMMA RESPONSE]:
 #include <iostream>
#include <vector>
#include <queue>
#include <limits>
using namespace std;
// Define the graph as an adjacency list
vector<vector<pair<int, int>>> graph;
// Define the priority queue for Dijkstra's algorithm
priority_queue<pair<int, int>, vector<pair<int, int>>, greater<pair<int, int>>> pq;
// Define the distance array to store the shortest distance from the source node
vector<int> dist;
// Define the visited array to keep track of visited nodes
vector<bool> visited;
// Define the function to perform Dijkstra's algorithm
void dijkstra(int src) {
// Initialize the distance array with infinity and the visited array with false
dist.assign(graph.size(), numeric_limits<int>::max());
visited.assign(graph.size(), false);
// Initialize the priority queue with the source node and its distance
pq.push({0, src});
// Perform Dijkstra's algorithm
while (!pq.empty()) {
// Get the node with the mi

In [6]:
import os
import json
from mlx_lm import load, stream_generate
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from duckduckgo_search import DDGS

# ==========================================
# 1. SYSTEM ARCHITECTURE & PATH DEFINITIONS
# ==========================================
# Routing I/O to external APFS Pendrive to save internal SSD wear
PD_BASE = "/Volumes/Macintosh PD/Gemma_Brain"
DB_DIR = os.path.join(PD_BASE, "chroma_db")
CODE_DIR = os.path.join(PD_BASE, "code_repository")
HISTORY_DIR = os.path.join(PD_BASE, "chat_history")
HISTORY_FILE = os.path.join(HISTORY_DIR, "session.json")

# Ensure directories exist on the pendrive
for path in [DB_DIR, CODE_DIR, HISTORY_DIR]:
    os.makedirs(path, exist_ok=True)

# ==========================================
# 2. MEMORY & EMBEDDINGS INIT (Cold Storage)
# ==========================================
print("--> Initializing Memory Embeddings (CPU optimized)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("--> Connecting to Vector Storage on Macintosh PD...")
vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

# ==========================================
# 3. INTERNET SEARCH ENGINE (Live Data)
# ==========================================
def search_web(query, max_results=3):
    """Scours the internet dynamically for real-time coding answers."""
    print(f"--> Scouring the web for real-time context...")
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        
        web_context = "LIVE INTERNET SEARCH RESULTS:\n"
        for r in results:
            web_context += f"Source: {r.get('href')}\nSnippet: {r.get('body')}\n\n"
        return web_context
    except Exception as e:
        print(f"--> [Web Search Failed: {e}]")
        return ""

# ==========================================
# 4. MLX COMPUTE ENGINE (Unified RAM)
# ==========================================
print("--> Loading Gemma 8-bit into M4 Unified Memory...")
# Replace "mlx_model" with your local folder path or HuggingFace repo (e.g., "google/gemma-4-e4b")
model, tokenizer = load("mlx_model") 

# ==========================================
# 5. CORE DEEPMIND PIPELINE
# ==========================================
def save_chat_history(user_query, ai_response):
    """Logs conversation state to the pendrive, bypassing RAM usage."""
    history =[]
    # If history file exists, load it first
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, "r") as f:
                history = json.load(f)
        except json.JSONDecodeError:
            pass # Ignore if file is empty or corrupted

    # Append new interaction
    history.append({"user": user_query, "model": ai_response})
    
    # Save back to pendrive
    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=4)

def coding_assistant(query, use_rag=True, use_web=False):
    """The Ultimate Generation Loop with Streaming, RAG, and Web Search."""
    
    context = ""
    
    # Check Pendrive Code Context
    if use_rag:
        # Failsafe in case DB is empty
        try:
            docs = vector_db.similarity_search(query, k=3)
            if docs:
                context += "LOCAL REPOSITORY CONTEXT (Macintosh PD):\n"
                for d in docs:
                    context += f"--- {d.metadata.get('source', 'Unknown')} ---\n{d.page_content}\n\n"
        except Exception:
            pass
            
    # Check Live Internet Context
    if use_web:
        web_results = search_web(query)
        if web_results:
            context += f"{web_results}\n"
            
    # Build the Gemma Prompt
    prompt = f"<bos><start_of_turn>user\n"
    if context:
        prompt += f"System Context Information:\n{context}\n"
    
    prompt += f"System Task: You are an expert C/C++/Python coding assistant.\nUser Query: {query}<end_of_turn>\n<start_of_turn>model\n"

    print("\n[GEMMA IS TYPING...]\n")
    
    # Generate and Stream Token-by-Token
    full_response = ""
    for response in stream_generate(model, tokenizer, prompt=prompt, max_tokens=1024):
        # Extract the actual text string from the GenerationResponse object
        text_chunk = response.text 
        
        print(text_chunk, end="", flush=True) 
        full_response += text_chunk
        
    print("\n\n[DONE]")
    
    # Offload to Pendrive
    save_chat_history(query, full_response)
    
    return full_response

# ==========================================
# 6. EXECUTION & TESTING
# ==========================================
# Test 1: Let's test the Web Search functionality to find latest API docs!
test_query = "What is the latest syntax to initialize a FastAPI server in Python? Give me a basic code example."
output = coding_assistant(test_query, use_rag=False, use_web=True)

--> Initializing Memory Embeddings (CPU optimized)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5616.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--> Connecting to Vector Storage on Macintosh PD...
--> Loading Gemma 8-bit into M4 Unified Memory...
--> Scouring the web for real-time context...


/var/folders/kh/qc595q4s4fxfxqgxjtcw1hkr0000gn/T/ipykernel_7597/3937553775.py:40: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



[GEMMA IS TYPING...]

The latest syntax to initialize a FastAPI server in Python is as follows:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
return {"Hello": "World"}
```

In this example, we import the `FastAPI` class from the `fastapi` module and create an instance of it called `app`. We then define a route using the `@app.get("/")` decorator, which specifies that the route should be a GET request to the root URL ("/"). The function `read_root()` is then defined to handle the request and return a JSON response with the message "Hello World".

To run the server, you can use the `uvicorn` command-line tool. First, install `uvicorn` using pip:

```
pip install uvicorn
```

Then, run the following command in your terminal:

```
uvicorn main:app --reload
```

This will start the server and automatically reload it when you make changes to the code. You can then access the server at `http://127.0.0.1:8000/` in your web browser.
<end_of_turn>
<star